In [2]:
# Import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

# Models
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Load dataset
data = pd.read_csv("heart.csv")

# -------------------------------
# Features and target
# -------------------------------
X = data.drop("HeartDisease", axis=1)
y = data["HeartDisease"]

# -------------------------------
# Column separation
# -------------------------------
categorical_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
numeric_cols = ["Age", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "Oldpeak"]

# -------------------------------
# Preprocessing
# -------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(), categorical_cols)
    ]
)

# -------------------------------
# Train-test split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------------
# Models
# -------------------------------
models = {
    "SVM": SVC(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier()
}

# -------------------------------
# BEFORE PCA
# -------------------------------
print("---- BEFORE PCA ----")
best_score = 0

for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("classifier", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    score = accuracy_score(y_test, y_pred)

    print(f"{name}: {score}")

    if score > best_score:
        best_score = score

print("Best Accuracy before PCA:", best_score)

# -------------------------------
# AFTER PCA
# -------------------------------
print("\n---- AFTER PCA ----")

pca = PCA(n_components=0.95)

best_score_pca = 0

for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("pca", pca),
        ("classifier", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    score = accuracy_score(y_test, y_pred)

    print(f"{name}: {score}")

    if score > best_score_pca:
        best_score_pca = score

print("Best Accuracy after PCA:", best_score_pca)

---- BEFORE PCA ----
SVM: 0.8641304347826086
Logistic Regression: 0.8532608695652174
Random Forest: 0.875
Best Accuracy before PCA: 0.875

---- AFTER PCA ----
SVM: 0.8586956521739131
Logistic Regression: 0.8532608695652174
Random Forest: 0.8369565217391305
Best Accuracy after PCA: 0.8586956521739131
